# Bring Your Own Schema

Skip schema discovery entirely. Define a Pydantic model, point catchfly at your documents, and get structured, normalized records back.

**What you'll learn:** User-provided schemas, `on_schema_ready` callback, JSON Schema dict input, category normalization.

**Estimated API cost:** ~$0.02

In [ ]:
# pip install "catchfly[openai,export]"
# export OPENAI_API_KEY="sk-..."

## Define your schema

This example extracts structured data from SaaS support tickets. Pydantic `Field(description=...)` directly improves extraction quality — the LLM uses these descriptions.

In [ ]:
from pydantic import BaseModel, Field


class SupportTicket(BaseModel):
    subject: str = Field(description="Brief summary of the issue")
    category: str = Field(description="e.g., Authentication, Billing, Bug, Feature Request")
    priority: str = Field(description="Critical, High, Medium, or Low")
    product_area: str = Field(description="Which product area is affected")
    customer_email: str | None = Field(default=None, description="Customer contact email if mentioned")
    is_resolved: bool = Field(default=False, description="Whether the issue appears resolved")

## Load data and run

In [ ]:
from catchfly import Pipeline
from catchfly.extraction import LLMDirectExtraction
from catchfly.normalization import LLMCanonicalization
from catchfly.demo import load_samples

docs = load_samples("support_tickets")

pipeline = Pipeline(
    extraction=LLMDirectExtraction(model="gpt-5.4-mini"),
    normalization=LLMCanonicalization(model="gpt-5.4-mini"),
)

results = pipeline.run(
    docs,
    schema=SupportTicket,
    normalize_fields=["category", "priority"],
)

print(f"Extracted {len(results.records)} tickets")

## Inspect extracted records

In [ ]:
for ticket in results.records[:5]:
    email = f" ({ticket.customer_email})" if ticket.customer_email else ""
    print(f"[{ticket.priority}] {ticket.category}: {ticket.subject}{email}")

## Normalization: messy categories to clean taxonomy

The raw tickets use inconsistent category labels. After normalization:

In [ ]:
cat_norm = results.normalizations["category"]
print("Category groups:")
for canonical, members in cat_norm.clusters.items():
    print(f"  {canonical}: {members}")

In [ ]:
pri_norm = results.normalizations["priority"]
print("Priority groups:")
for canonical, members in pri_norm.clusters.items():
    print(f"  {canonical}: {members}")

## Schema callback: inspect before extraction

Use `on_schema_ready` to review (and optionally modify) the schema before extraction begins:

In [ ]:
from catchfly._types import Schema


def review_schema(schema: Schema) -> Schema | None:
    print(f"Schema has {len(schema.json_schema['properties'])} fields:")
    for name in schema.json_schema["properties"]:
        print(f"  - {name}")
    # Return None to accept, or return a modified Schema to override
    return None


results = pipeline.run(
    docs,
    schema=SupportTicket,
    normalize_fields=["category"],
    on_schema_ready=review_schema,
)

## Using a JSON Schema dict instead of Pydantic

If your schema comes from an external source (API, database, config file), pass a raw JSON Schema dict:

In [ ]:
json_schema = {
    "title": "SupportTicket",
    "type": "object",
    "properties": {
        "subject": {"type": "string", "description": "Brief summary"},
        "category": {"type": "string"},
        "priority": {"type": "string", "enum": ["Critical", "High", "Medium", "Low"]},
    },
    "required": ["subject", "category", "priority"],
}

results = pipeline.run(docs, schema=json_schema, normalize_fields=["category"])
print(f"Records: {len(results.records)}")

## Export

In [ ]:
df = results.to_dataframe()
df[["subject", "category", "priority"]]